# Chapter 10 Student Colab Notebook: Numerical Methods

**Course:** MATH 3260 - Linear Algebra  
**Sections:** 10.1-10.4  
**Audience:** Student version

This notebook is a detailed computational companion for numerical linear algebra. It explains floating-point storage, rounding and conditioning, Gaussian elimination with partial pivoting, Jacobi and Gauss-Seidel iterations, the power method, Rayleigh quotients, polynomial regression, probability models, and stable age distributions. Every algorithm is derived step by step and accompanied by proofs, diagnostics, and Python verification.

> **Notebook conventions.** Display mathematics uses `$$...$$` for reliable Colab rendering. Numerical examples distinguish exact arithmetic, simulated finite-precision arithmetic, and ordinary double precision. The material is deliberately extensive; select portions for live class and retain the rest as a reference.

> **Student workflow.** Run each code cell, annotate the numerical tables, and complete each section worksheet before using the separate solution keys. All theory and proofs are retained.

## Chapter map

1. **10.1:** floating-point form, significant digits, absolute and relative error, Gaussian elimination, partial pivoting, conditioning, and residuals;
2. **10.2:** Jacobi and Gauss-Seidel iterations, matrix splittings, convergence, diagonal dominance, stopping rules, and divergence;
3. **10.3:** dominant eigenpairs, scaled power iteration, Rayleigh quotients, convergence rate, and failure modes;
4. **10.4:** quadratic regression with pivoting, Gauss-Seidel probability models, and Leslie population models with the power method.

In [ ]:
# Core imports and reusable numerical helpers
import math
from decimal import Decimal, ROUND_HALF_EVEN
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

sp.init_printing()

def round_sig(x, digits=4):
    """Round x to significant digits using round-half-to-even."""
    x=float(x)
    if x==0 or not np.isfinite(x):
        return x
    exponent=int(math.floor(math.log10(abs(x))))-digits+1
    quantum=Decimal(f'1e{exponent}')
    return float(Decimal(str(x)).quantize(quantum, rounding=ROUND_HALF_EVEN))

def gaussian_rounded(A,b,digits=4,partial_pivoting=False):
    """Gaussian elimination with rounding after every elementary arithmetic step."""
    M=[[round_sig(v,digits) for v in list(row)+[rhs]] for row,rhs in zip(A,b)]
    n=len(M); history=[('Initial augmented matrix',[r[:] for r in M])]
    for k in range(n-1):
        if partial_pivoting:
            p=max(range(k,n),key=lambda i:abs(M[i][k]))
            if abs(M[p][k])==0: raise ValueError('Singular matrix')
            if p!=k:
                M[k],M[p]=M[p],M[k]
                history.append((f'Swap R{k+1} and R{p+1}',[r[:] for r in M]))
        if abs(M[k][k])==0: raise ValueError('Zero pivot')
        for i in range(k+1,n):
            multiplier=round_sig(M[i][k]/M[k][k],digits)
            for j in range(k,n+1):
                product=round_sig(multiplier*M[k][j],digits)
                M[i][j]=round_sig(M[i][j]-product,digits)
            history.append((f'R{i+1} <- R{i+1} - ({multiplier})R{k+1}',[r[:] for r in M]))
    x=[0.0]*n
    for i in range(n-1,-1,-1):
        total=0.0
        for j in range(i+1,n):
            total=round_sig(total+round_sig(M[i][j]*x[j],digits),digits)
        x[i]=round_sig(round_sig(M[i][-1]-total,digits)/M[i][i],digits)
    return np.array(x),history,np.array(M,dtype=float)

def iteration_table(A,b,x0,iterations,method='jacobi'):
    A=np.array(A,dtype=float); b=np.array(b,dtype=float); x=np.array(x0,dtype=float)
    history=[x.copy()]
    for _ in range(iterations):
        if method=='jacobi':
            old=x.copy()
            x=np.array([(b[i]-A[i,:i]@old[:i]-A[i,i+1:]@old[i+1:])/A[i,i] for i in range(len(x))])
        else:
            for i in range(len(x)):
                x[i]=(b[i]-A[i,:i]@x[:i]-A[i,i+1:]@x[i+1:])/A[i,i]
        history.append(x.copy())
    return np.array(history)

def power_history(A,x0,iterations=8):
    A=np.array(A,dtype=float); x=np.array(x0,dtype=float)
    rows=[]
    for k in range(1,iterations+1):
        y=A@x
        scale=np.max(np.abs(y))
        x=y/scale
        rq=float(x@(A@x)/(x@x))
        rows.append([k,*x,scale,rq])
    return np.array(rows),x

def residual(A,x,b):
    return np.array(b,dtype=float)-np.array(A,dtype=float)@np.array(x,dtype=float)

# Section 10.1 - Gaussian Elimination with Partial Pivoting

## Learning objectives

- write decimal numbers in normalized floating-point form;
- distinguish truncation, rounding, absolute error, and relative error;
- explain why elementary arithmetic can amplify stored-value error;
- perform Gaussian elimination with simulated finite precision;
- implement partial pivoting and justify its pivot rule;
- distinguish a small residual from a small solution error;
- use the condition number to quantify sensitivity.

## Floating-point representation

A base-ten normalized floating-point number has the form

$$
\pm M\times 10^k,
\qquad 0.1\le M<1.
$$

For example,

$$
-4382.7=-0.43827\times 10^4.
$$

A machine with $t$ decimal significant digits stores only $t$ digits in the mantissa. The stored number is written $\operatorname{fl}(x)$.

**Round-half-to-even:** when the discarded part is exactly half a unit in the last retained place, choose the retained digit to be even. This avoids a systematic upward bias in long sequences of ties.

In [ ]:
values=[54.75,0.11345,-8.2255,0.08335,0.08345,98765.4]
rows=[]
for v in values:
    rows.append([v,round_sig(v,3),round_sig(v,4)])
display(pd.DataFrame(rows,columns=['exact','3 significant digits','4 significant digits']))

## Error measures

If $x$ is exact and $\widehat x$ is computed, then

$$
E_{\mathrm{abs}}=|x-\widehat x|,
$$

and, for $x\ne0$,

$$
E_{\mathrm{rel}}=\frac{|x-\widehat x|}{|x|}.
$$

Relative error compares the error with the scale of the quantity. An absolute error of $0.01$ is tiny for a distance of $1000$ miles but large for a clearance of $0.02$ inch.

### First-order floating-point model

For a correctly rounded operation and no overflow or underflow, a useful model is

$$
\operatorname{fl}(x\circ y)=(x\circ y)(1+\delta),
\qquad |\delta|\le u,
$$

where $u$ is the unit roundoff. This does not say all errors are harmless: many small local errors can be magnified by subtraction, large multipliers, and ill-conditioning.

## Catastrophic cancellation

Suppose $x$ and $y$ are close and already contain small storage errors. In

$$
x-y,
$$

leading digits cancel. The absolute error may stay small, but the true difference is also small, so its relative error can become very large. Elimination can create this situation when a poor pivot produces huge multipliers.

For example,

$$
1.234567-1.234566=0.000001.
$$

The leading digits cancel. Even if the stored inputs are accurate to several digits, the tiny difference may retain very little meaningful relative accuracy.


## Detailed original example: a poor first pivot

Consider the system

$$
\begin{aligned}
0.002x_1+2x_2-2x_3&=6.002,\\
3x_1-4x_2-4x_3&=-1,\\
x_1+4x_2-4x_3&=13.
\end{aligned}
$$

The exact solution is

$$
\mathbf{x}=(1,2,-1)^T.
$$

Without pivoting, the first multipliers are

$$
m_{21}=\frac{3}{0.002}=1500,
\qquad
m_{31}=\frac{1}{0.002}=500.
$$

These large multipliers magnify every error in the first row. With four-significant-digit rounding after every operation, naive elimination returns approximately

$$
\widehat{\mathbf{x}}=(2.000,2.199,-0.800)^T.
$$

In [ ]:
A=np.array([[0.002,2,-2],[3,-4,-4],[1,4,-4]],dtype=float)
x_exact=np.array([1,2,-1],dtype=float)
b=A@x_exact
x_naive,h_naive,U_naive=gaussian_rounded(A,b,4,False)
for title,M in h_naive:
    display(Markdown(f'**{title}**'))
    display(pd.DataFrame(M))
print('Naive rounded solution:',x_naive)
print('Residual:',residual(A,x_naive,b))

## Partial pivoting

At elimination stage $k$, partial pivoting chooses

$$
p=\arg\max_{i\ge k}|a_{ik}|
$$

and swaps rows $k$ and $p$.

### Why this reduces multiplier growth

After the swap, the pivot has largest magnitude in its active column. Therefore, for every row below it,

$$
|m_{ik}|=\left|\frac{a_{ik}}{a_{kk}}\right|\le1.
$$

Partial pivoting does not eliminate rounding error, but it prevents the immediate creation of multipliers such as $1500$ in the example.

**The fix to the small-pivot problem is to choose a better pivot before eliminating.** Partial pivoting implements that idea systematically.


In [ ]:
x_pivot,h_pivot,U_pivot=gaussian_rounded(A,b,4,True)
for title,M in h_pivot:
    display(Markdown(f'**{title}**'))
    display(pd.DataFrame(M))
print('Partial-pivoting solution:',x_pivot)
print('Exact solution:',x_exact)
print('Infinity-norm solution error:',np.linalg.norm(x_pivot-x_exact,np.inf))
print('Residual:',residual(A,x_pivot,b))

## Residual, forward error, and backward error

For a computed solution $\widehat{\mathbf{x}}$ of $A\mathbf{x}=\mathbf b$, the residual is

$$
\mathbf r=\mathbf b-A\widehat{\mathbf{x}}.
$$

- **Forward error:** $\mathbf{x}-\widehat{\mathbf{x}}$; it requires knowledge of the exact solution.
- **Residual:** measures how well the computed vector satisfies the original equations.
- **Backward error:** asks how much $A$ or $\mathbf b$ must change to make $\widehat{\mathbf{x}}$ exact.

A small residual does not by itself guarantee a small forward error if the system is ill-conditioned.

## Conditioning and a perturbation bound

### Definition: condition number

For an invertible matrix and a consistent norm,

$$
\kappa(A)=\|A\|\,\|A^{-1}\|.
$$

### Perturbation bound

Suppose $A\mathbf{x}=\mathbf b$ and $A(\mathbf{x}+\Delta\mathbf{x})=\mathbf b+\Delta\mathbf b$. Then

$$
A\Delta\mathbf{x}=\Delta\mathbf b,
\qquad
\Delta\mathbf{x}=A^{-1}\Delta\mathbf b.
$$

Therefore

$$
\|\Delta\mathbf{x}\|\le\|A^{-1}\|\,\|\Delta\mathbf b\|.
$$

Also $\|\mathbf b\|=\|A\mathbf{x}\|\le\|A\|\,\|\mathbf{x}\|$, so

$$
\frac1{\|\mathbf{x}\|}\le\frac{\|A\|}{\|\mathbf b\|}.
$$

Multiplying the inequalities gives

$$
\boxed{
\frac{\|\Delta\mathbf{x}\|}{\|\mathbf{x}\|}
\le
\kappa(A)
\frac{\|\Delta\mathbf b\|}{\|\mathbf b\|}
}.
$$

Thus the condition number is an error-amplification factor in the worst case.

In [ ]:
A_good=np.array([[4.,1.],[1.,3.]])
A_bad=np.array([[1.,1.],[1.,1.0001]])
for name,M in [('well-conditioned',A_good),('ill-conditioned',A_bad)]:
    print(name,'cond_2 =',np.linalg.cond(M,2))

## Section 10.1 terminology

| Term | Meaning |
|---|---|
| Significant digits | Retained meaningful digits in a stored number |
| Unit roundoff | Bound in the relative floating-point model |
| Cancellation | Loss of relative accuracy in subtracting nearby numbers |
| Partial pivoting | Largest-magnitude pivot selection in the active column |
| Residual | $\mathbf b-A\widehat{\mathbf x}$ |
| Forward error | Difference between exact and computed solutions |
| Condition number | Worst-case sensitivity factor $\|A\|\|A^{-1}\|$ |
| Ill-conditioned | Small input changes can cause large solution changes |

## Section 10.1 worksheet

1. Round $0.009995$, $18.245$, and $-763.55$ to four significant digits using round-half-to-even.
2. For the original $3\times3$ system above, reproduce every nonpivoted elimination step with four-significant-digit arithmetic.
3. Repeat with partial pivoting and compare the multipliers.
4. Compute residuals and forward errors for both approximations.
5. Prove the condition-number perturbation bound.
6. Explain why partial pivoting improves numerical stability but does not repair ill-conditioning.

# Section 10.2 - Iterative Methods for Linear Systems

## Stationary iterations

Write

$$
A=D+L+U,
$$

where $D$ is diagonal, $L$ strictly lower triangular, and $U$ strictly upper triangular.

### Jacobi

$$
D\mathbf{x}^{(k+1)}=\mathbf b-(L+U)\mathbf{x}^{(k)},
$$

so

$$
\mathbf{x}^{(k+1)}=B_J\mathbf{x}^{(k)}+\mathbf c_J,
\qquad
B_J=-D^{-1}(L+U).
$$

### Gauss-Seidel

$$
(D+L)\mathbf{x}^{(k+1)}=\mathbf b-U\mathbf{x}^{(k)},
$$

so

$$
\mathbf{x}^{(k+1)}=B_{GS}\mathbf{x}^{(k)}+\mathbf c_{GS},
\qquad
B_{GS}=-(D+L)^{-1}U.
$$

Gauss-Seidel immediately uses the newest available components.

Jacobi uses only values from the previous iterate. Gauss-Seidel immediately reuses the newest values, so improvements made early in an iteration influence later components immediately. This is why Gauss-Seidel often reduces the error faster.


### Why iterative methods?

Direct methods such as Gaussian elimination reach a solution through a finite algebraic process. Iterative methods instead generate approximations

$$
\mathbf x^{(0)},\mathbf x^{(1)},\mathbf x^{(2)},\ldots
$$

that aim to converge to the solution. For very large sparse systems, this can save both memory and computation. The tradeoff is that convergence must be analyzed.


## Error iteration and the spectral-radius criterion

Let $\mathbf{x}^*$ be the exact solution and let $\mathbf e^{(k)}=\mathbf{x}^{(k)}-\mathbf{x}^*$. Since the exact solution is a fixed point,

$$
\mathbf e^{(k+1)}=B\mathbf e^{(k)}=B^{k+1}\mathbf e^{(0)}.
$$

Therefore the method converges for every initial vector precisely when

$$
B^k\to0.
$$

For a finite-dimensional matrix,

$$
\boxed{B^k\to0\iff \rho(B)<1},
$$

where $\rho(B)$ is the maximum eigenvalue magnitude.

### Proof idea in the diagonalizable case

If $B=P\Lambda P^{-1}$, then

$$
B^k=P\Lambda^kP^{-1}.
$$

If every $|\lambda_i|<1$, then $\Lambda^k\to0$. If some $|\lambda_i|\ge1$, an eigenvector in that direction does not decay. The general Jordan-form proof reaches the same conclusion because polynomial factors cannot overcome exponential decay when $|\lambda|<1$.

## Detailed example

Solve iteratively

$$
\begin{aligned}
10x_1-x_2+2x_3&=6,\\
-x_1+11x_2-x_3&=22,\\
2x_1-x_2+10x_3&=-10.
\end{aligned}
$$

The exact solution is $(1,2,-1)^T$. The update equations are

$$
\begin{aligned}
x_1&=\frac{6+x_2-2x_3}{10},\\
x_2&=\frac{22+x_1+x_3}{11},\\
x_3&=\frac{-10-2x_1+x_2}{10}.
\end{aligned}
$$

### From matrix form to component form

A row

$$
a_{ii}x_i+\sum_{j\ne i}a_{ij}x_j=b_i
$$

can be solved for its diagonal variable:

$$
x_i=
\frac1{a_{ii}}
\left(
b_i-\sum_{j\ne i}a_{ij}x_j
\right).
$$

Doing this row by row produces the component update formulas.


In [ ]:
Ait=np.array([[10.,-1.,2.],[-1.,11.,-1.],[2.,-1.,10.]])
bit=np.array([6.,22.,-10.])
J=iteration_table(Ait,bit,[0,0,0],7,'jacobi')
G=iteration_table(Ait,bit,[0,0,0],5,'gs')
display(Markdown('**Jacobi iterates**'))
display(pd.DataFrame(J,columns=['x1','x2','x3']).round(7))
display(Markdown('**Gauss-Seidel iterates**'))
display(pd.DataFrame(G,columns=['x1','x2','x3']).round(7))
BJ=-np.diag(1/np.diag(Ait))@(Ait-np.diag(np.diag(Ait)))
BGS=-np.linalg.solve(np.tril(Ait),np.triu(Ait,1))
print('rho(B_J)=',max(abs(np.linalg.eigvals(BJ))))
print('rho(B_GS)=',max(abs(np.linalg.eigvals(BGS))))

In [ ]:
plt.figure(figsize=(7,4.5))
for name,H in [('Jacobi',J),('Gauss-Seidel',G)]:
    errors=np.linalg.norm(H-np.array([1,2,-1]),ord=np.inf,axis=1)
    plt.semilogy(range(len(errors)),errors,marker='o',label=name)
plt.xlabel('iteration'); plt.ylabel('infinity-norm error'); plt.title('Convergence comparison'); plt.grid(True,alpha=.3); plt.legend(); plt.show()

## Strict diagonal dominance

$A$ is strictly diagonally dominant by rows when

$$
|a_{ii}|>\sum_{j\ne i}|a_{ij}|
\qquad\text{for every }i.
$$

### Full Jacobi convergence proof

Define

$$
q=\max_i\frac{\sum_{j\ne i}|a_{ij}|}{|a_{ii}|}<1.
$$

From the component error equation,

$$
e_i^{(k+1)}=-\frac1{a_{ii}}\sum_{j\ne i}a_{ij}e_j^{(k)}.
$$

Hence

$$
|e_i^{(k+1)}|
\le
\frac{\sum_{j\ne i}|a_{ij}|}{|a_{ii}|}\|\mathbf e^{(k)}\|_\infty
\le q\|\mathbf e^{(k)}\|_\infty.
$$

Taking the maximum over $i$ gives

$$
\|\mathbf e^{(k+1)}\|_\infty\le q\|\mathbf e^{(k)}\|_\infty,
$$

and induction yields

$$
\|\mathbf e^{(k)}\|_\infty\le q^k\|\mathbf e^{(0)}\|_\infty\to0.
$$

### Full Gauss-Seidel convergence proof

For the $i$th updated component,

$$
e_i^{(k+1)}=-\frac1{a_{ii}}
\left(
\sum_{j<i}a_{ij}e_j^{(k+1)}+
\sum_{j>i}a_{ij}e_j^{(k)}
\right).
$$

Induct on $i$. Assume the already updated components satisfy
$|e_j^{(k+1)}|\le q\|\mathbf e^{(k)}\|_\infty$. Since $q<1$,

$$
\begin{aligned}
|e_i^{(k+1)}|
&\le\frac{q\sum_{j<i}|a_{ij}|+\sum_{j>i}|a_{ij}|}{|a_{ii}|}\|\mathbf e^{(k)}\|_\infty\\
&\le\frac{\sum_{j\ne i}|a_{ij}|}{|a_{ii}|}\|\mathbf e^{(k)}\|_\infty\\
&\le q\|\mathbf e^{(k)}\|_\infty.
\end{aligned}
$$

Thus Gauss-Seidel is also a contraction.

## Reordering a divergent system

For

$$
x+4y=5,
\qquad
3x+y=4,
$$

Jacobi in the displayed order uses

$$
x^{(k+1)}=5-4y^{(k)},
\qquad
y^{(k+1)}=4-3x^{(k)},
$$

and diverges from $(0,0)$. Reorder the equations:

$$
3x+y=4,
\qquad x+4y=5.
$$

The new coefficient matrix is strictly diagonally dominant, and both methods converge to $(1,1)$.

In [ ]:
Adiv=np.array([[1.,4.],[3.,1.]])
bdiv=np.array([5.,4.])
Are=np.array([[3.,1.],[1.,4.]])
bre=np.array([4.,5.])
display(pd.DataFrame(iteration_table(Adiv,bdiv,[0,0],6,'jacobi'),columns=['x','y']))
display(pd.DataFrame(iteration_table(Are,bre,[0,0],8,'jacobi'),columns=['x','y']).round(6))

## Stopping rules

A practical method may stop when

$$
\|\mathbf{x}^{(k+1)}-\mathbf{x}^{(k)}\|_\infty
\le \tau_{\mathrm{abs}}+\tau_{\mathrm{rel}}\|\mathbf{x}^{(k+1)}\|_\infty,
$$

and should also check

$$
\|\mathbf b-A\mathbf{x}^{(k+1)}\|_\infty.
$$

The change between iterates detects stagnation; the residual tests the equations. A maximum iteration count prevents an infinite loop.

### Direct versus iterative methods

| Situation | Typical preference |
|---|---|
| Small/medium dense system | direct factorization |
| Many right-hand sides, same $A$ | LU factorization |
| Very large sparse system | iterative method |
| Need a solve without convergence testing | direct method |
| Favorable spectral/diagonal structure | iterative method can be very efficient |


## Section 10.2 worksheet

1. Derive Jacobi and Gauss-Seidel update equations for the detailed $3\times3$ system.
2. Starting at zero, compute three iterations of each method by hand.
3. Compute $B_J$, $B_{GS}$, and their spectral radii.
4. Prove convergence from strict diagonal dominance.
5. Demonstrate divergence of the badly ordered $2\times2$ system and convergence after reordering.
6. Design a stopping rule using both iterate change and residual.

# Section 10.3 - Power Method for Dominant Eigenvalues

An eigenvalue $\lambda_1$ is dominant when

$$
|\lambda_1|>|\lambda_j|
\qquad(j\ge2).
$$

The corresponding eigenspace contains dominant eigenvectors.

## Derivation of the power method

Assume $A$ is diagonalizable with eigenbasis $\mathbf v_1,\ldots,\mathbf v_n$, and write

$$
\mathbf{x}^{(0)}=c_1\mathbf v_1+\cdots+c_n\mathbf v_n,
\qquad c_1\ne0.
$$

Then

$$
A^k\mathbf{x}^{(0)}
=c_1\lambda_1^k\mathbf v_1+\sum_{j=2}^n c_j\lambda_j^k\mathbf v_j.
$$

Factor out $\lambda_1^k$:

$$
A^k\mathbf{x}^{(0)}
=\lambda_1^k\left[
c_1\mathbf v_1+
\sum_{j=2}^n c_j\left(\frac{\lambda_j}{\lambda_1}\right)^k\mathbf v_j
\right].
$$

Every ratio tends to zero. After scaling, the direction approaches $\mathbf v_1$. The leading asymptotic rate is approximately

$$
\left|\frac{\lambda_2}{\lambda_1}\right|^k.
$$

### Goal of the power method

The power method estimates the **dominant eigenvalue** and a corresponding eigenvector without computing every eigenpair. It is especially useful when the matrix is large and the dominant long-run mode is the main quantity of interest.


## Scaled iteration

To avoid overflow or underflow, compute

$$
\mathbf y^{(k+1)}=A\mathbf x^{(k)},
\qquad
\alpha_{k+1}=\|\mathbf y^{(k+1)}\|_\infty,
\qquad
\mathbf x^{(k+1)}=\frac{\mathbf y^{(k+1)}}{\alpha_{k+1}}.
$$

Scaling changes magnitude but not direction.

## Detailed example

Use

$$
A=\begin{bmatrix}4&1\\2&3\end{bmatrix},
\qquad
\mathbf x^{(0)}=\begin{bmatrix}1\\0\end{bmatrix}.
$$

The eigenvalues are $5$ and $2$. The dominant eigenvector is a multiple of $(1,1)^T$.

In [ ]:
Ap=np.array([[4.,1.],[2.,3.]])
PH,xlast=power_history(Ap,[1,0],8)
display(pd.DataFrame(PH,columns=['k','x1','x2','scale','Rayleigh quotient']).round(9))

## Rayleigh quotient

For nonzero $\mathbf x$,

$$
R_A(\mathbf x)=\frac{\mathbf x^TA\mathbf x}{\mathbf x^T\mathbf x}.
$$

If $A\mathbf x=\lambda\mathbf x$, then

$$
R_A(\mathbf x)
=\frac{\mathbf x^T(\lambda\mathbf x)}{\mathbf x^T\mathbf x}
=\lambda.
$$

For an approximate eigenvector, the quotient gives an eigenvalue estimate. For symmetric $A$ it has especially strong accuracy properties and lies between the smallest and largest eigenvalues.

The normalization scale factor and the Rayleigh quotient are two eigenvalue estimates. The scale factor is cheap because normalization already computes it. The Rayleigh quotient uses the whole vector and is especially effective for symmetric matrices.


### Convergence-speed interpretation

The asymptotic speed is controlled mainly by

$$
\left|\frac{\lambda_2}{\lambda_1}\right|.
$$

A ratio near $0.1$ gives rapid convergence, while a ratio near $0.99$ gives slow convergence.


In [ ]:
# Visualize direction convergence
angles=[]
x=np.array([1.,0.])
v=np.array([1.,1.])/np.sqrt(2)
for k in range(9):
    xu=x/np.linalg.norm(x)
    angles.append(np.degrees(np.arccos(np.clip(abs(xu@v),-1,1))))
    y=Ap@x; x=y/np.max(np.abs(y))
plt.figure(figsize=(6.5,4))
plt.semilogy(range(9),np.maximum(angles,1e-12),marker='o')
plt.xlabel('iteration'); plt.ylabel('angle error (degrees)'); plt.title('Power-method direction error'); plt.grid(True,alpha=.3); plt.show()

## Failure and slow-convergence cases

1. If $c_1=0$, the initial vector has no dominant-eigenvector component.
2. If two eigenvalues share the largest magnitude, the direction may oscillate or remain in a multidimensional invariant subspace.
3. If $|\lambda_2/\lambda_1|$ is close to one, convergence is slow.
4. A negative dominant eigenvalue can cause sign alternation; direction convergence is interpreted up to sign.
5. A defective or strongly nonnormal matrix may show transient behavior not predicted by eigenvalue magnitude alone.

A reliable implementation monitors direction change, Rayleigh quotient change, and the eigen-residual

$$
\|A\mathbf x-R_A(\mathbf x)\mathbf x\|.
$$

## Section 10.3 worksheet

1. Find the exact eigenpairs of $A=\begin{bmatrix}4&1\\2&3\end{bmatrix}$.
2. Perform six scaled power iterations from $(1,0)^T$.
3. Compute the Rayleigh quotient and eigen-residual at each step.
4. Prove the power-method convergence formula for a diagonalizable matrix.
5. Compare matrices with eigenvalue ratios $0.2$ and $0.9$.
6. Give an initial vector for which the method misses the dominant eigenvalue.

# Section 10.4 - Applications of Numerical Methods

## Quadratic least-squares regression

For data $(x_i,y_i)$ and

$$
p(x)=a_0+a_1x+a_2x^2,
$$

minimize

$$
S(a_0,a_1,a_2)=\sum_{i=1}^n\left[y_i-(a_0+a_1x_i+a_2x_i^2)\right]^2.
$$

Setting all three partial derivatives to zero gives

$$
\begin{bmatrix}
n&\sum x_i&\sum x_i^2\\
\sum x_i&\sum x_i^2&\sum x_i^3\\
\sum x_i^2&\sum x_i^3&\sum x_i^4
\end{bmatrix}
\begin{bmatrix}a_0\\a_1\\a_2\end{bmatrix}
=
\begin{bmatrix}
\sum y_i\\\sum x_iy_i\\\sum x_i^2y_i
\end{bmatrix}.
$$

These are the normal equations $X^TX\mathbf a=X^T\mathbf y$. Partial pivoting is appropriate when solving them, although QR factorization is generally more stable for serious least-squares computation.

## Original hiking-elevation example

A fictional trail records elevation gain after hours $x=0,1,\ldots,5$:

$$
120,182,255,329,388,415.
$$

The needed sums are

$$
\sum x=15,
\quad\sum x^2=55,
\quad\sum x^3=225,
\quad\sum x^4=979,
$$

and

$$
\sum y=1689,
\quad\sum xy=5306,
\quad\sum x^2y=20746.
$$

Thus

$$
\begin{bmatrix}
6&15&55\\15&55&225\\55&225&979
\end{bmatrix}
\begin{bmatrix}a_0\\a_1\\a_2\end{bmatrix}
=
\begin{bmatrix}1689\\5306\\20746\end{bmatrix}.
$$

In [ ]:
xdata=np.arange(6,dtype=float)
ydata=np.array([120,182,255,329,388,415],dtype=float)
X=np.column_stack([np.ones_like(xdata),xdata,xdata**2])
a=np.linalg.solve(X.T@X,X.T@ydata)
print('a0,a1,a2 =',a)
xx=np.linspace(0,6,300)
print('prediction at x=6:',a@[1,6,36])
plt.figure(figsize=(7,4.5)); plt.scatter(xdata,ydata,label='observed'); plt.plot(xx,a[0]+a[1]*xx+a[2]*xx**2,label='quadratic fit'); plt.xlabel('hours'); plt.ylabel('elevation gain'); plt.title('Fictional hiking-elevation regression'); plt.grid(True,alpha=.3); plt.legend(); plt.show()


## Probability as a harmonic linear system

At an interior state, the probability of eventual success equals the weighted average of the success probabilities at the next states. Consider four trail intersections with success probabilities $p_1,\ldots,p_4$:

$$
p_1=\frac12p_2,
$$

$$
p_2=\frac13(p_1+p_3+p_4),
$$

$$
p_3=\frac12(p_2+1),
$$

$$
p_4=\frac13(p_2+1+0).
$$

The constants $1$ and $0$ represent immediate success and failure exits. The exact solution is

$$
(p_1,p_2,p_3,p_4)=\left(\frac14,\frac12,\frac34,\frac12\right).
$$

Gauss-Seidel is natural because each newly updated probability is used immediately.

In [ ]:
p=np.zeros(4); ph=[p.copy()]
for _ in range(10):
    p[0]=p[1]/2
    p[1]=(p[0]+p[2]+p[3])/3
    p[2]=(p[1]+1)/2
    p[3]=(p[1]+1)/3
    ph.append(p.copy())
display(pd.DataFrame(ph,columns=['p1','p2','p3','p4']).round(8))

## Leslie matrix and a stable age distribution

For age classes, a Leslie matrix has fertility coefficients in the first row and survival probabilities on the subdiagonal. Use the fictional trail-songbird matrix

$$
L=\begin{bmatrix}
0&1.1&1.6&0.5\\
0.72&0&0&0\\
0&0.65&0&0\\
0&0&0.55&0
\end{bmatrix}.
$$

If $\mathbf n_k$ is the age-count vector, then

$$
\mathbf n_{k+1}=L\mathbf n_k.
$$

A stable age distribution has constant proportions, so

$$
L\mathbf v=\lambda\mathbf v.
$$

The dominant eigenvalue is the long-run per-period growth factor; a normalized positive dominant eigenvector gives the stable proportions.

In [ ]:
L=np.array([[0.,1.1,1.6,.5],[.72,0,0,0],[0,.65,0,0],[0,0,.55,0]])
rows,v=power_history(L,[1,1,1,1],25)
proportions=np.abs(v)/np.abs(v).sum()
lam=float(v@(L@v)/(v@v))
print('dominant growth factor ~',lam)
print('stable proportions ~',proportions)
plt.figure(figsize=(7,4.5))
# recompute normalized proportions at each step
x=np.ones(4); H=[]
for k in range(20):
    x=L@x; x=x/x.sum(); H.append(x.copy())
H=np.array(H)
for j in range(4): plt.plot(range(1,21),H[:,j],marker='o',markersize=3,label=f'class {j+1}')
plt.xlabel('iteration'); plt.ylabel('population proportion'); plt.title('Convergence to stable age proportions'); plt.grid(True,alpha=.3); plt.legend(); plt.show()

## Model limitations

Numerical output is conditional on the model:

- regression extrapolation can be unreliable outside the data interval;
- maze probabilities assume fixed transition probabilities and eventual absorption;
- Leslie models assume time-invariant fertility and survival and ignore density dependence;
- convergence of an algorithm does not prove that the underlying scientific assumptions are correct.

A responsible analysis reports algorithmic error, data uncertainty, and model limitations separately.

## Section 10.4 worksheet

1. Derive the quadratic normal equations from the sum of squared residuals.
2. Solve the hiking-elevation normal equations with partial pivoting and predict at $x=6$.
3. Derive the four probability equations from the transition diagram described above.
4. Perform eight Gauss-Seidel iterations and compare with the exact solution.
5. Use the power method to estimate the trail-songbird growth factor and stable proportions.
6. Explain at least three limitations of each application model.

# Chapter synthesis

Numerical linear algebra studies how mathematical formulas behave when implemented with finite storage and finite computation.

1. Partial pivoting controls elimination multipliers and usually improves stability.
2. Conditioning belongs to the problem; stability belongs to the algorithm.
3. Jacobi and Gauss-Seidel are fixed-point iterations whose errors evolve by matrix powers.
4. Strict diagonal dominance gives a concrete contraction proof.
5. The power method isolates the dominant eigendirection at a rate governed by an eigenvalue ratio.
6. Regression, probability, and population models reuse the same matrix algorithms in different applications.
7. Residuals, stopping rules, and model assumptions must be checked—not merely the final number.